In [9]:
import os
os.getcwd()

'C:\\Users\\eugen'

In [10]:
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv("universal_top_spotify_songs.csv")

In [12]:
df.head()   # muestra las primeras 5 filas

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,2RkZ5LkEzeHGRsmDqKwmaJ,Ordinary,Alex Warren,1,1,0,NaN,2025-06-11,95,False,...,2,-6.141,1,0.0600,0.704000,0.000007,0.0550,0.391,168.115,3
1,42UBPzRMh5yyz0EDPr6fr1,Manchild,Sabrina Carpenter,2,-1,48,NaN,2025-06-11,89,True,...,7,-5.087,1,0.0572,0.122000,0.000000,0.3170,0.811,123.010,4
2,0FTmksd2dxiE5e3rWyJXs6,back to friends,sombr,3,0,1,NaN,2025-06-11,98,False,...,1,-2.291,1,0.0301,0.000094,0.000088,0.0929,0.235,92.855,4
3,7so0lgd0zP2Sbgs2d7a1SZ,Die With A Smile,"Lady Gaga, Bruno Mars",4,0,-1,NaN,2025-06-11,91,False,...,6,-7.727,0,0.0317,0.289000,0.000000,0.1260,0.498,157.964,3
4,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,5,1,0,NaN,2025-06-11,100,False,...,2,-10.171,1,0.0358,0.200000,0.060800,0.1170,0.438,104.978,4


In [8]:
# ============================================
# CONFIGURACIÓN
# ============================================
EXPORTAR_LIMPIO = True  # Cambia a False si NO quieres exportar el CSV
TAMANO_MUESTRA = 100000  # Cantidad de canciones para la muestra fina

In [13]:
# ============================================
# CARGAR DATOS
# ============================================
print("📀 Cargando dataset...")
df = pd.read_csv('universal_top_spotify_songs.csv')
print(f"✅ Dataset cargado: {len(df):,} canciones")

📀 Cargando dataset...
✅ Dataset cargado: 2,110,316 canciones


In [14]:
# ============================================
# LAS 11 FEATURES QUE USAREMOS (según tu tabla)
# ============================================
features = [
    'danceability',    # float (0-1) - Qué tan bailable es
    'energy',          # float (0-1) - Intensidad/actividad
    'valence',         # float (0-1) - Positividad/alegría
    'acousticness',    # float (0-1) - Acústica vs electrónica
    'instrumentalness', # float (0-1) - Si no tiene voz
    'liveness',        # float (0-1) - Si tiene audiencia en vivo
    'speechiness',     # float (0-1) - Cantidad de palabras habladas
    'tempo',           # float (BPM) - Velocidad de la canción
    'loudness',        # float (dB) - Volumen (negativo a positivo)
    'duration_ms',     # int (ms) - Duración
    'mode'             # int (0 o 1) - Mayor o menor
]

In [11]:
# Verificar que todas las columnas existen en el dataset
columnas_faltantes = [col for col in features if col not in df.columns]
if columnas_faltantes:
    print(f"⚠️ Atención: Estas columnas NO existen en el dataset: {columnas_faltantes}")
else:
    print(f"✅ Las 11 features existen en el dataset")

✅ Las 11 features existen en el dataset


In [15]:
# ============================================
# 1. DIAGNÓSTICO COMPLETO
# ============================================
print("\n" + "="*60)
print("🔍 DIAGNÓSTICO COMPLETO DEL DATASET")
print("="*60)


🔍 DIAGNÓSTICO COMPLETO DEL DATASET


In [13]:
# 1.1 Valores nulos
print("\n📌 1. VALORES NULOS:")
print("-"*40)
nulos = df[features].isnull().sum()
if nulos.sum() == 0:
    print("✅ No hay valores nulos en ninguna feature")
else:
    print(nulos[nulos > 0])


📌 1. VALORES NULOS:
----------------------------------------
✅ No hay valores nulos en ninguna feature


In [16]:
# 1.2 Valores imposibles
print("\n📌 2. VALORES IMPOSIBLES:")
print("-"*40)
tempo_cero = df[df['tempo'] == 0]
print(f"⚠️ tempo = 0: {len(tempo_cero)} canciones")


📌 2. VALORES IMPOSIBLES:
----------------------------------------
⚠️ tempo = 0: 1 canciones


In [15]:
# Verificar mode (debe ser 0 o 1)
mode_fuera = df[(df['mode'] != 0) & (df['mode'] != 1)]
if len(mode_fuera) > 0:
    print(f"⚠️ mode fuera de rango (0 o 1): {len(mode_fuera)} canciones")

In [17]:
# Verificar duration_ms > 0
duration_cero = df[df['duration_ms'] <= 0]
if len(duration_cero) > 0:
    print(f"⚠️ duration_ms <= 0: {len(duration_cero)} canciones")

⚠️ duration_ms <= 0: 29 canciones


In [18]:
# 1.3 Duplicados exactos
print("\n📌 3. DUPLICADOS EXACTOS (misma fila en todas las columnas):")
print("-"*40)
duplicados = df.duplicated().sum()
print(f"⚠️ Filas duplicadas: {duplicados}")


📌 3. DUPLICADOS EXACTOS (misma fila en todas las columnas):
----------------------------------------
⚠️ Filas duplicadas: 0


In [19]:
# 1.4 Tipos de datos
print("\n📌 4. TIPOS DE DATOS:")
print("-"*40)
for col in features:
    print(f"   {col}: {df[col].dtype}")


📌 4. TIPOS DE DATOS:
----------------------------------------
   danceability: float64
   energy: float64
   valence: float64
   acousticness: float64
   instrumentalness: float64
   liveness: float64
   speechiness: float64
   tempo: float64
   loudness: float64
   duration_ms: int64
   mode: int64


In [20]:
# ============================================
# 2. LIMPIEZA
# ============================================
print("\n" + "="*60)
print("🧹 APLICANDO LIMPIEZA")
print("="*60)


🧹 APLICANDO LIMPIEZA


In [21]:
df_limpio = df.copy()

In [22]:
# Eliminar tempo = 0
antes = len(df_limpio)
df_limpio = df_limpio[df_limpio['tempo'] > 0]
print(f"✅ Eliminadas {antes - len(df_limpio)} canciones con tempo = 0")

✅ Eliminadas 1 canciones con tempo = 0


In [25]:
# Eliminar mode que no sea 0 o 1 (si existe)
antes = len(df_limpio)
df_limpio = df_limpio[(df_limpio['mode'] == 0) | (df_limpio['mode'] == 1)]
print(f"✅ Eliminadas {antes - len(df_limpio)} canciones con mode inválido")

✅ Eliminadas 0 canciones con mode inválido


In [23]:
# Eliminar duration_ms <= 0
antes = len(df_limpio)
df_limpio = df_limpio[df_limpio['duration_ms'] > 0]
print(f"✅ Eliminadas {antes - len(df_limpio)} canciones con duration_ms <= 0")

✅ Eliminadas 29 canciones con duration_ms <= 0


In [27]:
# Eliminar duplicados exactos
antes = len(df_limpio)
df_limpio = df_limpio.drop_duplicates()
print(f"✅ Eliminadas {antes - len(df_limpio)} filas duplicadas")

✅ Eliminadas 0 filas duplicadas


In [24]:
# Eliminar nulos en las features
antes = len(df_limpio)
df_limpio = df_limpio.dropna(subset=features)
print(f"✅ Eliminadas {antes - len(df_limpio)} canciones con valores nulos")

print(f"\n📊 RESULTADO FINAL: {len(df_limpio):,} canciones listas para clustering")


✅ Eliminadas 0 canciones con valores nulos

📊 RESULTADO FINAL: 2,110,286 canciones listas para clustering


In [31]:
EXPORTAR_LIMPIO = True

In [37]:
# ============================================
# 3. EXPORTAR CSV LIMPIO (OPCIONAL)
# ============================================
if EXPORTAR_LIMPIO:
    print("\n" + "="*60)
    print("💾 EXPORTANDO DATASET LIMPIO")
    print("="*60)
    
    # Exportar completo (puede ser pesado)
    df_limpio.to_csv('universal_top_spotify_songs_clean.csv', index=False)
    print("✅ Exportado: universal_top_spotify_songs_clean.csv")
    
    # Exportar solo una muestra (más rápido)
    df_features = df_limpio[features]

    df_features.sample(n=50000, random_state=42).to_csv(
        'spotify_features_sample.csv',
        index=False
    )

    print("✅ Exportado: spotify_features_sample.csv (muestra de 50k)")

else:
    print("\n⏭️ Exportación desactivada (EXPORTAR_LIMPIO = False)")


💾 EXPORTANDO DATASET LIMPIO
✅ Exportado: universal_top_spotify_songs_clean.csv
✅ Exportado: spotify_features_sample.csv (muestra de 50k)


In [40]:
# ============================================
# 4. MUESTRA PARA CLUSTERING
# ============================================
print("\n" + "="*60)
print("🎯 MUESTRA PARA CLUSTERING")
print("="*60)


🎯 MUESTRA PARA CLUSTERING


In [39]:
# Tomar muestra aleatoria
df_muestra = df_limpio.sample(n=min(TAMANO_MUESTRA, len(df_limpio)), random_state=42)

In [45]:
# Escalar los datos (necesario para clustering)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_muestra[features])
print(f"✅ Muestra tomada: {len(df_muestra):,} canciones")
print(f"✅ Features escaladas: {X_scaled.shape[1]} variables")
print(f"✅ Escalador guardado (usar mismo para nuevos datos)")

✅ Muestra tomada: 2,000 canciones
✅ Features escaladas: 11 variables
✅ Escalador guardado (usar mismo para nuevos datos)


In [46]:
# Exportar muestra
if EXPORTAR_LIMPIO:
    # Muestra original (sin escalar)
    df_muestra.to_csv('spotify_muestra.csv', index=False)
    print("✅ Exportado: spotify_muestra.csv (datos sin escalar)")
    
    # Muestra escalada
    df_muestra_scaled = pd.DataFrame(X_scaled, columns=features)
    df_muestra_scaled.to_csv('spotify_muestra_scaled.csv', index=False)
    print("✅ Exportado: spotify_muestra_scaled.csv (datos escalados)")

✅ Exportado: spotify_muestra.csv (datos sin escalar)
✅ Exportado: spotify_muestra_scaled.csv (datos escalados)


In [43]:
# ============================================
# 5. RESUMEN FINAL
# ============================================
print("\n" + "="*60)
print("📋 RESUMEN FINAL")
print("="*60)
print(f"""
─────────────────────────────────────────────────────────────
\tDATASET ORIGINAL:\t{2_110_316:>12,} canciones
\tDATASET LIMPIO:\t{len(df_limpio):>12,} canciones
\tMUESTRA PARA ML:\t{len(df_muestra):>12,} canciones
\tFEATURES USADAS:\t{len(features):>12} variables
─────────────────────────────────────────────────────────────
\tLISTA DE FEATURES:
\t1. danceability     7. liveness
\t2. energy           8. speechiness
\t3. valence          9. tempo
\t4. acousticness     10. loudness
\t5. instrumentalness 11. duration_ms
\t6. mode
─────────────────────────────────────────────────────────────
\t✅ Dataset listo para aplicar los 3 algoritmos:
\t\t- K-Means
\t\t- DBSCAN
\t\t- Agglomerative Clustering
─────────────────────────────────────────────────────────────
""")



📋 RESUMEN FINAL

─────────────────────────────────────────────────────────────
	DATASET ORIGINAL:	   2,110,316 canciones
	DATASET LIMPIO:	   2,110,286 canciones
	MUESTRA PARA ML:	       2,000 canciones
	FEATURES USADAS:	          11 variables
─────────────────────────────────────────────────────────────
	LISTA DE FEATURES:
	1. danceability     7. liveness
	2. energy           8. speechiness
	3. valence          9. tempo
	4. acousticness     10. loudness
	5. instrumentalness 11. duration_ms
	6. mode
─────────────────────────────────────────────────────────────
	✅ Dataset listo para aplicar los 3 algoritmos:
		- K-Means
		- DBSCAN
		- Agglomerative Clustering
─────────────────────────────────────────────────────────────



In [36]:
# ============================================
# 6. ESTADÍSTICAS DE LA MUESTRA
# ============================================
print("\n📊 ESTADÍSTICAS DE LA MUESTRA (datos sin escalar):")
print("-"*40)
print(df_muestra[features].describe().round(3))


📊 ESTADÍSTICAS DE LA MUESTRA (datos sin escalar):
----------------------------------------
       danceability      energy     valence  acousticness  instrumentalness  \
count    100000.000  100000.000  100000.000    100000.000        100000.000   
mean          0.676       0.648       0.545         0.277             0.023   
std           0.144       0.169       0.231         0.252             0.113   
min           0.094       0.000       0.000         0.000             0.000   
25%           0.581       0.551       0.368         0.068             0.000   
50%           0.700       0.668       0.546         0.192             0.000   
75%           0.779       0.767       0.731         0.438             0.000   
max           0.988       0.993       0.985         0.996             0.995   

         liveness  speechiness       tempo    loudness  duration_ms  \
count  100000.000   100000.000  100000.000  100000.000   100000.000   
mean        0.170        0.096     122.060      -6.770

In [47]:
# ============================================
# 7. VERIFICACIÓN FINAL DE RANGOS
# ============================================
print("\n📌 VERIFICACIÓN DE RANGOS ESPERADOS:")
print("-"*40)

rangos = {
    'danceability': (0, 1),
    'energy': (0, 1),
    'valence': (0, 1),
    'acousticness': (0, 1),
    'instrumentalness': (0, 1),
    'liveness': (0, 1),
    'speechiness': (0, 1),
    'tempo': (0, 300),
    'loudness': (-60, 10),
    'duration_ms': (1000, 600000),  # 1 segundo a 10 minutos
    'mode': (0, 1)
}

for feature, (min_esp, max_esp) in rangos.items():
    valores = df_muestra[feature]
    fuera = valores[(valores < min_esp) | (valores > max_esp)]
    if len(fuera) > 0:
        print(f"⚠️ {feature}: {len(fuera)} valores fuera de rango [{min_esp}, {max_esp}]")
    else:
        print(f"✅ {feature}: todos en rango [{min_esp}, {max_esp}]")


📌 VERIFICACIÓN DE RANGOS ESPERADOS:
----------------------------------------
✅ danceability: todos en rango [0, 1]
✅ energy: todos en rango [0, 1]
✅ valence: todos en rango [0, 1]
✅ acousticness: todos en rango [0, 1]
✅ instrumentalness: todos en rango [0, 1]
✅ liveness: todos en rango [0, 1]
✅ speechiness: todos en rango [0, 1]
✅ tempo: todos en rango [0, 300]
✅ loudness: todos en rango [-60, 10]
✅ duration_ms: todos en rango [1000, 600000]
✅ mode: todos en rango [0, 1]


In [48]:
from sklearn.cluster import AgglomerativeClustering

In [50]:
TAMANO_MUESTRA = 2000  # ideal: 2000–5000

In [49]:
df_muestra = df_limpio.sample(n=2000, random_state=42)

In [53]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_muestra[features])

In [54]:
from sklearn.decomposition import PCA

pca = PCA(n_components=5)
X_reducido = pca.fit_transform(X_scaled)

In [55]:
# Crear modelo
from sklearn.cluster import AgglomerativeClustering

modelo = AgglomerativeClustering(
    n_clusters=5,
    linkage='ward'
)

In [56]:
#Aplicando el clustering
clusters = modelo.fit_predict(X_reducido)

df_muestra['cluster'] = clusters
print(df_muestra['cluster'].value_counts())

cluster
1    792
0    656
2    455
3     77
4     20
Name: count, dtype: int64


In [58]:
#Interpretar los cluster 
df_clusters = df_muestra.groupby('cluster')[features].mean().round(3)
print(df_clusters)

         danceability  energy  valence  acousticness  instrumentalness  \
cluster                                                                  
0               0.642   0.734    0.555         0.163             0.006   
1               0.761   0.668    0.632         0.256             0.002   
2               0.574   0.482    0.378         0.493             0.009   
3               0.716   0.601    0.477         0.173             0.405   
4               0.292   0.291    0.223         0.695             0.565   

         liveness  speechiness    tempo  loudness  duration_ms   mode  
cluster                                                                
0           0.256        0.096  133.783    -5.495   184406.684  0.648  
1           0.111        0.121  116.640    -6.249   188004.518  0.327  
2           0.136        0.057  115.098    -8.785   213912.360  0.774  
3           0.115        0.077  119.934    -9.799   234086.740  0.390  
4           0.277        0.234  104.813   -34.582

In [59]:
#Cluster 0
#danceability = 0.642  → alto
#energy       = 0.734  → alto
#tempo        = 133    → rápido
#acousticness = 0.163  → bajo
#---bailable + energética + rápida + no acústica
